# Kuwaiti Arabic — OpenAI Realtime API Evaluation

Tests the **Realtime API** (`gpt-realtime-2.1`) on `KW001` only. This complements
`kuwaiti_arabic_eval.ipynb`, which used request/response endpoints
(`gpt-4o-transcribe` + `gpt-audio` + `gpt-4o-mini`) and so could not measure
realistic time-to-first-audio.

## Flow

1. Decode the test-case MP3 to 24 kHz mono PCM16 (the Realtime API's audio format).
2. Open a Realtime WebSocket session with `client.realtime.connect(...)`; configure
   instructions (Kuwaiti Arabic clinic receptionist), voice `marin`, PCM in/out, input transcription
   (`gpt-4o-transcribe`, `language="ar"`), and **turn detection off** so we control
   exactly when the "utterance ends".
3. Append the audio, then `commit` + `response.create`. The stopwatch starts here.
4. Record **time to first audio** (first `response.output_audio.delta`) and **total
   response time** (`response.done`), and collect the audio deltas into a WAV file.

Note: the audio is uploaded in one burst rather than at real-time pace, and upload
time is excluded from the stopwatch. Time-to-first-audio therefore reflects the
model's turn-around after end of speech, which is what a phone agent's caller feels.
Input transcription runs asynchronously and is guidance, not exactly what the model heard.

In [1]:
import asyncio
import base64
import csv
import re
import time
import wave
from pathlib import Path

import miniaudio
from dotenv import find_dotenv, load_dotenv
from IPython.display import Audio, display
from openai import AsyncOpenAI

load_dotenv(find_dotenv(usecwd=True))  # never print the API key

REALTIME_MODEL = "gpt-realtime-2.1"
VOICE = "marin"
SAMPLE_RATE = 24000

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = REPO_ROOT / "kuwaiti_arabic_test_cases.csv"
AUDIO_DIR = REPO_ROOT / "audio" / "kuwaiti"
RESULTS_PATH = REPO_ROOT / "results" / "kuwaiti_realtime_results.csv"
RESPONSE_AUDIO_DIR = REPO_ROOT / "results" / "audio" / "kuwaiti"


## 1. Load the test case (KW001 only)

In [2]:
with open(CSV_PATH, encoding="utf-8-sig") as f:
    case = next(row for row in csv.DictReader(f) if row["id"] == "KW001")
audio_path = AUDIO_DIR / f"{case['id']}.mp3"
assert audio_path.exists(), audio_path

print("Test case:")
for k in ("id", "dialect", "category", "text", "expected_intent", "expected_entities"):
    print(f"  {k}: {case[k] or '(none)'}")
print("Customer audio:")
display(Audio(filename=str(audio_path)))


Test case:
  id: KW001
  dialect: kuwaiti
  category: greeting
  text: السلام عليكم
  expected_intent: greeting
  expected_entities: (none)
Customer audio:


## 2. Run one Realtime session

In [3]:
INSTRUCTIONS = (
    "You are a customer-service voice agent for a clinic that speaks Kuwaiti "
    "Arabic. Reply naturally and briefly in Kuwaiti Arabic, as a warm agent would "
    "on the phone. Do not translate to English or switch to Modern Standard Arabic."
)


def load_pcm16(path: Path) -> bytes:
    """Decode an MP3 to 24 kHz mono signed 16-bit PCM."""
    decoded = miniaudio.decode_file(
        str(path), output_format=miniaudio.SampleFormat.SIGNED16,
        nchannels=1, sample_rate=SAMPLE_RATE,
    )
    return decoded.samples.tobytes()


async def run_realtime(pcm: bytes) -> dict:
    client = AsyncOpenAI()
    result = {"input_transcript": "", "response_transcript": "", "audio": bytearray(),
              "first_audio_ms": None, "total_ms": None, "events": []}
    async with client.realtime.connect(model=REALTIME_MODEL) as conn:
        await conn.session.update(session={
            "type": "realtime",
            "output_modalities": ["audio"],
            "instructions": INSTRUCTIONS,
            "audio": {
                "input": {
                    "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
                    "transcription": {"model": "gpt-4o-transcribe", "language": "ar"},
                    "turn_detection": None,  # we decide when the utterance ends
                },
                "output": {"format": {"type": "audio/pcm", "rate": SAMPLE_RATE}, "voice": VOICE},
            },
        })
        chunk = SAMPLE_RATE * 2  # 1 second of PCM16 per append
        for i in range(0, len(pcm), chunk):
            await conn.input_audio_buffer.append(audio=base64.b64encode(pcm[i:i + chunk]).decode())

        t0 = time.perf_counter()  # stopwatch: end of customer utterance
        await conn.input_audio_buffer.commit()
        await conn.response.create()

        async for ev in conn:
            now_ms = round((time.perf_counter() - t0) * 1000)
            result["events"].append((now_ms, ev.type))
            if ev.type == "response.output_audio.delta":
                if result["first_audio_ms"] is None:
                    result["first_audio_ms"] = now_ms
                result["audio"] += base64.b64decode(ev.delta)
            elif ev.type == "conversation.item.input_audio_transcription.completed":
                result["input_transcript"] = ev.transcript
            elif ev.type == "response.output_audio_transcript.done":
                result["response_transcript"] = ev.transcript
            elif ev.type == "error":
                raise RuntimeError(ev.error)
            elif ev.type == "response.done":
                result["total_ms"] = now_ms
                break
    return result


# Notebooks already run an event loop, so await directly instead of asyncio.run().
rt = await run_realtime(load_pcm16(audio_path))

RESPONSE_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
response_audio_path = RESPONSE_AUDIO_DIR / (case["id"] + "_realtime_response.wav")
with wave.open(str(response_audio_path), "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(SAMPLE_RATE)
    w.writeframes(bytes(rt["audio"]))
audio_seconds = len(rt["audio"]) / (SAMPLE_RATE * 2)


## 3. Result: test case vs. model

In [4]:
print("=" * 60); print("TEST CASE"); print("=" * 60)
print(f"ID:              {case['id']}   Dialect: {case['dialect']}   Category: {case['category']}")
print(f"Expected text:   {case['text']}")
print(f"Expected intent: {case['expected_intent']}")
print(f"Expected entities: {case['expected_entities'] or '(none)'}")
print(); print("=" * 60); print(f"REALTIME RESULT  ({REALTIME_MODEL}, voice={VOICE})"); print("=" * 60)
print(f"Input transcription: {rt['input_transcript']}")
print(f"Agent response:      {rt['response_transcript']}")
print(f"Response audio:      {response_audio_path.relative_to(REPO_ROOT)}  ({audio_seconds:.1f} s)")
print()
print(f"Time to first audio: {rt['first_audio_ms']} ms")
print(f"Total response time: {rt['total_ms']} ms")
print()
print("Customer said:"); display(Audio(filename=str(audio_path)))
print("Agent replied (Realtime):"); display(Audio(filename=str(response_audio_path)))


TEST CASE
ID:              KW001   Dialect: kuwaiti   Category: greeting
Expected text:   السلام عليكم
Expected intent: greeting
Expected entities: (none)

REALTIME RESULT  (gpt-realtime-2.1, voice=marin)
Input transcription: السلام عليكم
Agent response:      وعليكم السلام حياك الله، شلون أقدر أساعدك اليوم؟
Response audio:      results/audio/kuwaiti/KW001_realtime_response.wav  (5.2 s)

Time to first audio: 985 ms
Total response time: 1810 ms

Customer said:


Agent replied (Realtime):


## 4. Simple evaluation

Transcription correctness is a normalized exact match (placeholder for full WER, see
spec §9.1). Intent/entity classification is not repeated here, since the Realtime session
only produces speech. Rate the response by ear: set `response_quality` (1–5) and `notes`.

In [5]:
def normalize_arabic(text: str) -> str:
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]", "", text)
    text = re.sub(r"[\u060C\u061B\u061F.,!?;:\"'ـ]", "", text)
    return re.sub(r"\s+", " ", text).strip()


transcription_correct = normalize_arabic(rt["input_transcript"]) == normalize_arabic(case["text"])
print("transcription_correct:", transcription_correct)

# --- Manual: fill in after listening ---
response_quality = None  # 1 (Poor) - 5 (Excellent)
notes = ""


transcription_correct: True


## 5. Save to `results/kuwaiti_realtime_results.csv`

In [6]:
fieldnames = ["id", "dialect", "category", "model", "expected_text", "input_transcription",
              "transcription_correct", "response", "response_audio_path", "response_audio_seconds",
              "time_to_first_audio_ms", "total_response_ms", "response_quality", "notes"]
row = {
    "id": case["id"], "dialect": case["dialect"], "category": case["category"],
    "model": REALTIME_MODEL, "expected_text": case["text"],
    "input_transcription": rt["input_transcript"], "transcription_correct": transcription_correct,
    "response": rt["response_transcript"],
    "response_audio_path": response_audio_path.relative_to(REPO_ROOT).as_posix(),
    "response_audio_seconds": round(audio_seconds, 2),
    "time_to_first_audio_ms": rt["first_audio_ms"], "total_response_ms": rt["total_ms"],
    "response_quality": response_quality, "notes": notes,
}
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
existing = []
if RESULTS_PATH.exists():
    with open(RESULTS_PATH, encoding="utf-8") as f:
        existing = [r for r in csv.DictReader(f) if r["id"] != case["id"]]
with open(RESULTS_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(existing + [row])
print("Saved", case["id"], "to", RESULTS_PATH)


Saved KW001 to /workspaces/arabic-voice-agent-evaluation/results/kuwaiti_realtime_results.csv
